In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, IntegerType, LongType, StringType, StructField, StructType, TimestampType,
)

if "batch_id" not in dbutils.widgets.getAll():
    dbutils.widgets.text("batch_id", "full-sample-20261010-01", "Bronze batch ID")
batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    raise ValueError("batch_id is required")

bronze_batch_df = (
    spark.table("workspace.tmdb_lakehouse.bronze_movie_records")
    .filter(F.col("batch_id") == batch_id)
)
bronze_count = bronze_batch_df.count()
if bronze_count == 0:
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")

named_ref_schema = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
])

child_movie_schema = StructType([
    StructField("id", LongType(), True),
    StructField("genres", ArrayType(named_ref_schema), True),
    StructField("production_companies", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("origin_country", StringType(), True),
    ])), True),
    StructField("credits", StructType([StructField("cast", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("credit_id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("character", StringType(), True),
        StructField("order", IntegerType(), True),
    ])), True)]), True),
    StructField("keywords", StructType([
        StructField("keywords", ArrayType(named_ref_schema), True),
    ]), True),
])

parsed_child_movies_df = (
    bronze_batch_df
    .select(
        "movie_id", "batch_id", "load_type", "source_file", "source_extracted_at",
        F.from_json("raw_payload", child_movie_schema).alias("movie"),
    )
    .select(
        "movie_id", "batch_id", "load_type",
        "source_file", "source_extracted_at", "movie.*",
    )
)

bad_ids = parsed_child_movies_df.filter(
    F.col("id").isNull() | (F.col("id") != F.col("movie_id"))
).count()
print(f"Bronze rows: {bronze_count}")
print(f"Parsed movie rows: {parsed_child_movies_df.count()}")
print(f"Missing or mismatched parsed IDs: {bad_ids}")

if bad_ids:
    raise ValueError(f"{bad_ids} Bronze records have a missing or mismatched movie ID")

In [0]:
CHILD_TABLES = {
    "silver_movie_genres": [
        ("movie_id", "BIGINT"), ("genre_id", "BIGINT"), ("genre_name", "STRING"),
    ],
    "silver_movie_studios": [
        ("movie_id", "BIGINT"), ("company_id", "BIGINT"),
        ("company_name", "STRING"), ("origin_country", "STRING"),
    ],
    "silver_movie_cast": [
        ("movie_id", "BIGINT"), ("credit_id", "STRING"), ("person_id", "BIGINT"),
        ("cast_name", "STRING"), ("character", "STRING"), ("cast_order", "INT"),
    ],
    "silver_movie_keywords": [
        ("movie_id", "BIGINT"), ("keyword_id", "BIGINT"), ("keyword_name", "STRING"),
    ],
}

lineage_columns = [
    ("source_batch_id", "STRING"), ("source_file", "STRING"),
    ("source_extracted_at", "TIMESTAMP"), ("row_hash", "STRING"),
    ("load_timestamp", "TIMESTAMP"),
]

for table_name, columns in CHILD_TABLES.items():
    column_sql = ", ".join(f"{name} {data_type}" for name, data_type in columns + lineage_columns)
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.{table_name} ({column_sql}) USING DELTA"
    )

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_child_snapshot_state (
    movie_id BIGINT, collection_name STRING, source_extracted_at TIMESTAMP,
    source_batch_id STRING, load_timestamp TIMESTAMP
) USING DELTA
""")

# Each collection: source array, child key, column mapping, required fields, hash fields.
COLLECTIONS = {
    "genres": {
        "table": "silver_movie_genres", "array": "genres", "key": "genre_id",
        "columns": [("id", "genre_id"), ("name", "genre_name")],
        "required": ["genre_id", "genre_name"],
        "hash": ["movie_id", "genre_id", "genre_name"],
    },
    "studios": {
        "table": "silver_movie_studios", "array": "production_companies", "key": "company_id",
        "columns": [("id", "company_id"), ("name", "company_name"), ("origin_country", "origin_country")],
        "required": ["company_id", "company_name"],
        "hash": ["movie_id", "company_id", "company_name", "origin_country"],
    },
    "cast": {
        "table": "silver_movie_cast", "array": "credits.cast", "key": "credit_id",
        "columns": [
            ("credit_id", "credit_id"), ("id", "person_id"), ("name", "cast_name"),
            ("character", "character"), ("order", "cast_order"),
        ],
        "required": ["credit_id"],
        "hash": ["movie_id", "credit_id", "person_id", "cast_name", "character", "cast_order"],
    },
    "keywords": {
        "table": "silver_movie_keywords", "array": "keywords.keywords", "key": "keyword_id",
        "columns": [("id", "keyword_id"), ("name", "keyword_name")],
        "required": ["keyword_id", "keyword_name"],
        "hash": ["movie_id", "keyword_id", "keyword_name"],
    },
}

log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("input_parameter", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_details", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])


def write_child_run_log(run_id, started_at, collection_name, status,
                        inserted, updated, quarantined=0, error_details=None):
    finished_at = datetime.now(timezone.utc).replace(tzinfo=None)
    load_type = parsed_child_movies_df.select("load_type").first()["load_type"]
    row = [(
        run_id, "Bronze-to-Silver", load_type,
        f"batch_id={batch_id}; collection={collection_name}",
        batch_id, started_at, finished_at, status,
        inserted, updated, quarantined, error_details, finished_at,
    )]
    (
        spark.createDataFrame(row, schema=log_schema)
        .write.format("delta").mode("append")
        .saveAsTable("workspace.tmdb_lakehouse.pipeline_execution_logs")
    )


def prepare_collection(name, cfg):
    """Explode one collection into rows and return its snapshot candidates."""
    present_df = parsed_child_movies_df.filter(F.col(cfg["array"]).isNotNull())

    # A present empty array is a valid snapshot and can clear stale rows.
    snapshot_candidates_df = present_df.select(
        "movie_id", "source_extracted_at",
        F.col("batch_id").alias("source_batch_id"),
        F.size(cfg["array"]).alias("item_count"),
    )

    source_df = (
        present_df
        .select(
            "movie_id", "batch_id", "source_file", "source_extracted_at",
            F.explode(cfg["array"]).alias("item"),
        )
        .select(
            "movie_id",
            *[F.col(f"item.{src}").alias(dst) for src, dst in cfg["columns"]],
            F.col("batch_id").alias("source_batch_id"),
            "source_file", "source_extracted_at",
        )
    )

    invalid_df = source_df.filter(
        F.col("movie_id").isNull()
        | F.expr(" OR ".join(f"`{column}` IS NULL" for column in cfg["required"]))
    )
    if invalid_df.limit(1).count():
        display(invalid_df.limit(20))
        raise ValueError(f"Found {invalid_df.count()} {name} rows missing a required field")

    conflicting_df = (
        source_df.groupBy("movie_id", cfg["key"])
        .agg(F.countDistinct(F.to_json(F.struct(*cfg["hash"]))).alias("versions"))
        .filter(F.col("versions") > 1)
    )
    if conflicting_df.limit(1).count():
        display(conflicting_df.limit(20))
        raise ValueError(f"Found {conflicting_df.count()} {name} keys with conflicting attributes")

    source_df = (
        source_df
        .dropDuplicates(["movie_id", cfg["key"]])
        .withColumn("row_hash", F.sha2(F.to_json(F.struct(*cfg["hash"])), 256))
        .withColumn("load_timestamp", F.current_timestamp())
    )

    print(
        f"{name}: movies={snapshot_candidates_df.count()}, "
        f"empty={snapshot_candidates_df.filter(F.col('item_count') == 0).count()}, "
        f"missing={parsed_child_movies_df.filter(F.col(cfg['array']).isNull()).count()}, "
        f"rows={source_df.count()}"
    )
    return snapshot_candidates_df, source_df


def merge_child_collection(collection_name, table_name, snapshot_candidates_df,
                           source_df, child_key):
    """Merge one collection, delete stale rows, and record the processed snapshot."""
    state_table = "workspace.tmdb_lakehouse.silver_child_snapshot_state"
    keys = ["movie_id", child_key]
    run_id, started_at = str(uuid4()), datetime.now(timezone.utc).replace(tzinfo=None)
    inserted = updated = deleted = quarantined = 0

    try:
        duplicate_snapshots = (
            snapshot_candidates_df.groupBy("movie_id").count()
            .filter("count > 1").count()
        )
        if duplicate_snapshots:
            raise ValueError(f"Found {duplicate_snapshots} duplicate {collection_name} snapshots")

        duplicate_keys = source_df.groupBy(*keys).count().filter("count > 1").count()
        if duplicate_keys:
            raise ValueError(
                f"Found {duplicate_keys} duplicate keys in the {collection_name} source"
            )

        stored_state_df = (
            spark.table(state_table)
            .filter(F.col("collection_name") == collection_name)
            .select(
                "movie_id",
                F.col("source_extracted_at").alias("stored_extracted_at"),
                F.col("source_batch_id").alias("stored_batch_id"),
            )
        )
        eligible_snapshots_df = (
            snapshot_candidates_df.alias("incoming")
            .join(stored_state_df.alias("stored"), on="movie_id", how="left")
            .filter(
                F.col("stored_batch_id").isNull()
                | (F.col("incoming.source_extracted_at") > F.col("stored_extracted_at"))
                | (
                    (F.col("incoming.source_extracted_at") == F.col("stored_extracted_at"))
                    & (F.col("incoming.source_batch_id") == F.col("stored_batch_id"))
                )
            )
            .select("movie_id", "source_extracted_at", "source_batch_id")
        )

        candidate_count = snapshot_candidates_df.count()
        eligible_count = eligible_snapshots_df.count()
        print(f"Eligible {collection_name} movie snapshots:", eligible_count)
        if eligible_count != candidate_count:
            raise ValueError(
                f"Some {collection_name} snapshots are older than stored data "
                "or have an ambiguous timestamp. Resolve snapshot ordering before processing."
            )

        eligible_ids_df = eligible_snapshots_df.select("movie_id").distinct()
        eligible_source_df = source_df.join(eligible_ids_df, on="movie_id", how="inner")
        existing_df = spark.table(table_name)

        if set(existing_df.columns) != set(eligible_source_df.columns):
            raise ValueError(
                f"{collection_name} source/table columns differ; "
                f"missing={sorted(set(existing_df.columns) - set(eligible_source_df.columns))}, "
                f"extra={sorted(set(eligible_source_df.columns) - set(existing_df.columns))}"
            )

        inserted = eligible_source_df.join(existing_df, keys, "left_anti").count()
        updated = (
            eligible_source_df.alias("incoming")
            .join(
                existing_df.alias("stored"),
                [F.col(f"incoming.{key}") == F.col(f"stored.{key}") for key in keys],
                "inner",
            )
            .filter(
                ~F.col("incoming.row_hash").eqNullSafe(F.col("stored.row_hash"))
                | ~F.col("incoming.source_extracted_at").eqNullSafe(F.col("stored.source_extracted_at"))
                | ~F.col("incoming.source_batch_id").eqNullSafe(F.col("stored.source_batch_id"))
                | ~F.col("incoming.source_file").eqNullSafe(F.col("stored.source_file"))
            )
            .count()
        )
        stale_rows_df = (
            existing_df.join(eligible_ids_df, on="movie_id", how="inner")
            .join(eligible_source_df.select(*keys), keys, "left_anti")
            .select(*keys).dropDuplicates()
        )
        deleted = stale_rows_df.count()

        suffix = run_id.replace("-", "")
        source_view = f"_{collection_name}_source_{suffix}"
        stale_view = f"_{collection_name}_stale_{suffix}"
        state_view = f"_{collection_name}_state_{suffix}"
        eligible_source_df.createOrReplaceTempView(source_view)
        stale_rows_df.createOrReplaceTempView(stale_view)

        quoted = [f"`{key}`" for key in keys]
        merge_condition = " AND ".join(f"target.{key} = source.{key}" for key in quoted)
        changed_condition = " OR ".join(
            f"NOT (target.`{column}` <=> source.`{column}`)"
            for column in ["row_hash", "source_extracted_at", "source_batch_id", "source_file"]
        )
        assignments = ", ".join(
            f"target.`{column}` = source.`{column}`"
            for column in existing_df.columns if column != "load_timestamp"
        ) + (
            ", target.`load_timestamp` = CASE WHEN NOT (target.`row_hash` <=> source.`row_hash`)"
            " THEN source.`load_timestamp` ELSE target.`load_timestamp` END"
        )
        insert_columns = ", ".join(f"`{column}`" for column in existing_df.columns)
        insert_values = ", ".join(f"source.`{column}`" for column in existing_df.columns)

        spark.sql(f"""
            MERGE INTO {table_name} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED AND ({changed_condition}) THEN UPDATE SET {assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
        """)
        spark.sql(f"""
            MERGE INTO {table_name} AS target
            USING {stale_view} AS stale
            ON target.{quoted[0]} = stale.{quoted[0]}
               AND target.{quoted[1]} = stale.{quoted[1]}
            WHEN MATCHED THEN DELETE
        """)

        eligible_snapshots_df.select(
            "movie_id",
            F.lit(collection_name).alias("collection_name"),
            "source_extracted_at", "source_batch_id",
            F.current_timestamp().alias("load_timestamp"),
        ).createOrReplaceTempView(state_view)
        spark.sql(f"""
            MERGE INTO {state_table} AS target
            USING {state_view} AS source
            ON target.movie_id = source.movie_id
               AND target.collection_name = source.collection_name
            WHEN MATCHED AND source.source_extracted_at > target.source_extracted_at
            THEN UPDATE SET
                target.source_extracted_at = source.source_extracted_at,
                target.source_batch_id = source.source_batch_id,
                target.load_timestamp = source.load_timestamp
            WHEN NOT MATCHED THEN INSERT (
                movie_id, collection_name, source_extracted_at, source_batch_id, load_timestamp
            ) VALUES (
                source.movie_id, source.collection_name, source.source_extracted_at,
                source.source_batch_id, source.load_timestamp
            )
        """)

        write_child_run_log(
            run_id, started_at, collection_name, "SUCCESS",
            inserted, updated, quarantined,
        )
        print(
            f"{collection_name.title()} inserted={inserted}, updated={updated}, "
            f"stale deleted={deleted}, stored={spark.table(table_name).count()}"
        )
        return {
            "inserted": inserted, "updated": updated,
            "deleted": deleted, "quarantined": quarantined,
        }

    except Exception as exc:
        try:
            write_child_run_log(
                run_id, started_at, collection_name, "FAILURE",
                inserted, updated, quarantined, str(exc)[:4000],
            )
        except Exception as log_exc:
            raise RuntimeError(
                f"{collection_name.title()} processing failed: {exc}; "
                f"failure logging also failed: {log_exc}"
            ) from log_exc
        raise

print("Child-table helpers defined.")

In [0]:
for collection_name, config in COLLECTIONS.items():
    snapshot_candidates_df, source_df = prepare_collection(collection_name, config)
    merge_child_collection(
        collection_name=collection_name,
        table_name=f"workspace.tmdb_lakehouse.{config['table']}",
        snapshot_candidates_df=snapshot_candidates_df,
        source_df=source_df,
        child_key=config["key"],
    )

In [0]:
spark.sql("""
SELECT run_id, batch_id, input_parameter, status,
       rows_inserted, rows_updated, rows_quarantined, error_details
FROM workspace.tmdb_lakehouse.pipeline_execution_logs
WHERE layer = 'Bronze-to-Silver'
  AND input_parameter LIKE '%; collection=%'
ORDER BY start_time DESC
LIMIT 20
""").show(truncate=False)